## Describe your model -> fine-tuned LLaMA 2
By Matt Shumer (https://twitter.com/mattshumer_)

The goal of this notebook is to experiment with a new way to make it very easy to build a task-specific model for your use-case.

First, use the best GPU available (go to Runtime -> change runtime type)

To create your model, just go to the first code cell, and describe the model you want to build in the prompt. Be descriptive and clear.

Select a temperature (high=creative, low=precise), and the number of training examples to generate to train the model. From there, just run all the cells.

You can change the model you want to fine-tune by changing `model_name` in the `Define Hyperparameters` cell.

In [1]:
import os
import sys

try:
    import pyarrow
    from datasets import load_dataset
    import trl
    import peft
    import bitsandbytes
    import accelerate
    import transformers
    print('✅ All training libraries are successfully installed and compatible! No restart required.')
except Exception as e:
    print(f'⚠️ Import check failed: {e}')
    print('Installing required libraries...')
    from IPython import get_ipython
    ipython = get_ipython()

    # We install WITHOUT the -U flag. This prevents pip from upgrading pre-installed datasets/pyarrow,
    # which keeps the environment stable and avoids binary incompatibility restarts.
    ipython.run_line_magic('pip', 'install -q accelerate peft bitsandbytes transformers trl')

    print('Checking compatibility after installation...')
    try:
        import trl
        from datasets import load_dataset
        print('✅ Imports successful! No restart required.')
    except Exception as e2:
        print(f'⚠️ Mismatch detected after install: {e2}')
        print('Upgrading pyarrow and datasets to resolve binary conflicts...')
        ipython.run_line_magic('pip', 'install -q -U pyarrow datasets')
        print()
        print('================================================================================')
        print('🔄 RESTARTING RUNTIME TO APPLY CHANGES...')
        print('The runtime will now restart automatically to apply C-extension updates.')
        print('👉 PLEASE CLICK "Run All" (or "Runtime" -> "Run all") ONE MORE TIME AFTER THE RESTART!')
        print('================================================================================')
        print()
        import time
        time.sleep(2)
        os.kill(os.getpid(), 9)

⚠️ Import check failed: No module named 'trl'
Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.4 MB/s eta 0:00:00
Checking compatibility after installation...
✅ Imports successful! No restart required.


In [2]:
import os

# Set base data directory (use Google Drive if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/gpt-llm-trainer"
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"📁 Google Drive mounted successfully. Data will be saved to: {DATA_DIR}")
except ImportError:
    DATA_DIR = "."
    print(f"📁 Local environment detected. Data will be saved to: {DATA_DIR}")

# Global paths
checkpoint_path = os.path.join(DATA_DIR, "checkpoint_examples.json")
train_path = os.path.join(DATA_DIR, "train.jsonl")
test_path = os.path.join(DATA_DIR, "test.jsonl")

Mounted at /content/drive
📁 Google Drive mounted successfully. Data will be saved to: /content/drive/MyDrive/gpt-llm-trainer


We also need to generate a system message.

Now let's put our examples into a dataframe and turn them into a final pair of datasets.

Split into train and test sets.

# Install necessary libraries

In [3]:

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# Define Hyperparameters

In [4]:
prompt = "A model that takes in an unstructured, raw conversation transcript between a doctor and a patient, and outputs a highly professional, structured clinical SOAP note containing Subjective, Objective, Assessment, and Plan sections."
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # use this if you have access to the official LLaMA 2 model "meta-llama/Llama-2-7b-chat-hf", though keep in mind you'll need to pass a Hugging Face key argument
dataset_name = "train.jsonl"
new_model = "llama-2-7b-custom"
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False
output_dir = "./results"
num_train_epochs = 1
fp16 = True
bf16 = False
per_device_train_batch_size = 2
per_device_eval_batch_size = 4
gradient_accumulation_steps = 4
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "constant"
max_steps = -1
warmup_ratio = 0.03
group_by_length = True
save_steps = 25
logging_steps = 5
max_seq_length = 1024
packing = False
device_map = {"": 0}

system_message = "You are a highly skilled medical professional specializing in transcribing doctor-patient conversations into structured clinical SOAP notes. Your task is to accurately extract relevant information from the dialogue and format it into Subjective, Objective, Assessment, and Plan sections. Maintain a professional, concise, and medically accurate tone."

#Load Datasets and Train

In [32]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

# Set environment variables
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

# Override training hyperparameters for high-quality convergence
num_train_epochs = 10
gradient_accumulation_steps = 2
bf16 = True
fp16 = False
bnb_4bit_compute_dtype = 'bfloat16'

# Initialize tokenizer first so we can use it for truncation during mapping
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load datasets
train_dataset = load_dataset('json', data_files=train_path, split="train")
valid_dataset = load_dataset('json', data_files=test_path, split="train")

# Preprocess datasets by truncating transcripts to 600 tokens
# This ensures that the prompt AND response fit within the 1024 token training limit,
# allowing the model to actually learn the transition to the SOAP note target.
def preprocess_function(examples):
    processed_texts = []
    for prompt_text, response_text in zip(examples['prompt'], examples['response']):
        transcript_tokens = tokenizer(prompt_text, truncation=True, max_length=600)['input_ids']
        truncated_transcript = tokenizer.decode(transcript_tokens, skip_special_tokens=True)
        full_text = f"[INST] <<SYS>>\n{system_message.strip()}\n<</SYS>>\n\n{truncated_transcript} [/INST] {response_text}"
        processed_texts.append(full_text)
    return {'text': processed_texts}

train_dataset_mapped = train_dataset.map(preprocess_function, batched=True)
valid_dataset_mapped = valid_dataset.map(preprocess_function, batched=True)

# Remove original columns to prevent TRL from auto-detecting conversational format
train_dataset_mapped = train_dataset_mapped.remove_columns(['prompt', 'response'])
valid_dataset_mapped = valid_dataset_mapped.remove_columns(['prompt', 'response'])

compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)
# Set training parameters using SFTConfig (modern TRL API)
training_arguments = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    report_to="none",
    eval_strategy="steps",
    eval_steps=10,  # Evaluate every 10 steps
    max_length=max_seq_length,
    dataset_text_field="text",
    packing=packing,
)
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_mapped,
    eval_dataset=valid_dataset_mapped,  # Pass validation dataset here
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments,
)
trainer.train()
trainer.model.save_pretrained(new_model)

# Test the model using the first validation example (full transcript)
logging.set_verbosity(logging.CRITICAL)
test_example = valid_dataset[0]
test_transcript = test_example['prompt']

# Truncate the transcript to fit within the model's context limit
input_ids = tokenizer(test_transcript, truncation=True, max_length=600)['input_ids']
truncated_transcript = tokenizer.decode(input_ids, skip_special_tokens=True)

prompt = f"[INST] <<SYS>>\n{system_message}\n<</SYS>>\n\n{truncated_transcript} [/INST]"

# Run generation with robust sampling parameters to prevent loops
gen = pipeline('text-generation', model=model, tokenizer=tokenizer)
result = gen(prompt, max_new_tokens=500, do_sample=True, temperature=0.3, top_p=0.9, repetition_penalty=1.2)
print(result[0]['generated_text'].replace(prompt, ''))


Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_1639/3512050451.py:78: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_arguments = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '1.765', 'grad_norm': '0.1406', 'learning_rate': '0.0002', 'entropy': '1.689', 'mean_token_accuracy': '0.5981', 'num_tokens': '2.048e+04', 'epoch': '0.4348'}
{'loss': '1.696', 'grad_norm': '0.1582', 'learning_rate': '0.0002', 'entropy': '1.693', 'mean_token_accuracy': '0.6125', 'num_tokens': '4.096e+04', 'epoch': '0.8696'}
{'eval_loss': '1.662', 'eval_runtime': '5.384', 'eval_samples_per_second': '0.929', 'eval_steps_per_second': '0.186', 'eval_entropy': '1.682', 'eval_mean_token_accuracy': '0.6211', 'eval_num_tokens': '4.096e+04', 'epoch': '0.8696'}
{'loss': '1.617', 'grad_norm': '0.1729', 'learning_rate': '0.0002', 'entropy': '1.688', 'mean_token_accuracy': '0.6225', 'num_tokens': '5.837e+04', 'epoch': '1.261'}
{'loss': '1.557', 'grad_norm': '0.1855', 'learning_rate': '0.0002', 'entropy': '1.623', 'mean_token_accuracy': '0.6358', 'num_tokens': '7.885e+04', 'epoch': '1.696'}
{'eval_loss': '1.543', 'eval_runtime': '5.392', 'eval_samples_per_second': '0.927', 'eval_steps_per_se

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Subjective:
- Chief Complaint: Worsening shortness of breath and progressive edema over several weeks; hypertensive patient with uncontrolled diabetes, CKD stage 3–4, type 2 diabetes mellitus, and prior coronary artery disease (CAD), presenting with acute dyspnea and tachycardia.
- History of Present Illness:
  - 68-year-old male with long-standing type II diabetes mellitus, CKD stages 3–4, HFrEF (EF ~30%), prior CAD, and history of hyperglycemic crisis, presents with acute dyspnea due to progressive edema and worsening symptoms of CAD-related dyslipidemias and hypertension.
  - Etiology:
    - Dyspnea: Previously described as “short of breath” after exertion. Currently reports “gasping for air.”
    - Swelling: Chronic systemic inflammatory response leading to increased vascular permeability and subsequent accumulation of extracellular fluids, including plasma, albumin, and red blood cells, resulting in edematous liver, spleen, and skeletal muscles.
    - Ankle swelling: Persistent sw

#Run Inference

In [33]:
from transformers import pipeline

system_message = "You are a highly skilled medical professional specializing in transcribing doctor-patient conversations into structured clinical SOAP notes. Your task is to accurately extract relevant information from the dialogue and format it into Subjective, Objective, Assessment, and Plan sections. Maintain a professional, concise, and medically accurate tone."

# Use the first full transcript from your test/validation dataset
test_example = valid_dataset[0]
test_transcript = test_example['prompt']

prompt = f"[INST] <<SYS>>\n{system_message}\\n<</SYS>>\n\n{test_transcript} [/INST]"

num_new_tokens = 500  # Number of new tokens to generate

# Count the number of tokens in the prompt

# Calculate the maximum length for the generation

gen = pipeline('text-generation', model=model, tokenizer=tokenizer)
result = gen(prompt, max_new_tokens=num_new_tokens)
print(result[0]['generated_text'].replace(prompt, ''))

s Doctor “s ‘s ‘s ‘<siced “s “s “s ‘s____ ‘s _ ‘bury surg clin [‘siced Slo—pat sug SW or or or<spat sugar SO medical sug and sever cut slash SO ors sli socket S squss subs surg slisbemataps‘ssb/spat sli sli API surg sap sli surg sub patients Ster subss subs sand slis Ssss sug socket sss patients Slos S S so sub sub diagn S S S S S so Sss subs Subs subs DR surg subcut subs sub Sub or subs diagn Seb S medicine surg subs subs doctor so subs Sub subs sub patient S sub subs Sub sub sub sub surg sand SO Sed surg subs S surg s subs subs sub sub sub sub Sub Sub Sub S Subs s sub S S sed surg ss SO subs sub subs sub S sub S and ss sap sub sed s S surg ss surg Ss sedss S sub surg ssync sters SO s CDs patient Patcut SO sap subs sub sub subsubs subs SOsspat s s SO patientspatsssss subs subs patients SOsss subssssss SO Soss sub subss sedssssbs patientpat patient soss SO Ss SO patient patientpatpatpat patient SO SOSO SO SO So doctor patient S so patient SO SO SO So sli patient patient Spat pat S pati

#Merge the model and store in Google Drive

In [14]:
!pip install -q -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.4 MB/s eta 0:00:00


In [34]:
# Merge and save the fine-tuned model
from google.colab import drive
drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/llama-2-7b-custom"  # change to your preferred path

# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Save the merged model
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/llama-2-7b-custom/tokenizer_config.json',
 '/content/drive/MyDrive/llama-2-7b-custom/chat_template.jinja',
 '/content/drive/MyDrive/llama-2-7b-custom/tokenizer.json')

# Load a fine-tuned model from Drive and run inference

In [38]:
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer

drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/llama-2-7b-custom"  # change to the path where your model is saved

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [39]:
from transformers import pipeline

system_message = "You are a highly skilled medical professional specializing in transcribing doctor-patient conversations into structured clinical SOAP notes. Your task is to accurately extract relevant information from the dialogue and format it into Subjective, Objective, Assessment, and Plan sections. Maintain a professional, concise, and medically accurate tone."

# Use the proper instruction format that the model was trained on
prompt = f"[INST] <<SYS>>\\n{system_message}\\n<</SYS>>\\n\\nDoctor: Hello, how can I help you today?\\nPatient: Hi, I've had a sore throat and fever for the last two days. [/INST]"

gen = pipeline('text-generation', model=model, tokenizer=tokenizer, max_length=500)
result = gen(prompt)
print(result[0]['generated_text'])

[INST] <<SYS>>\nYou are a highly skilled medical professional specializing in transcribing doctor-patient conversations into structured clinical SOAP notes. Your task is to accurately extract relevant information from the dialogue and format it into Subjective, Objective, Assessment, and Plan sections. Maintain a professional, concise, and medically accurate tone.\n<</SYS>>\n\nDoctor: Hello, how can I help you today?\nPatient: Hi, I've had a sore throat and fever for the last two days. [/INST]<</SYS>>">


In [40]:
# Load the first actual full conversation from the validation dataset
test_example = valid_dataset[0]
test_transcript = test_example['prompt']

# Truncate the transcript to 600 tokens to fit model limits
input_ids = tokenizer(test_transcript, truncation=True, max_length=600)['input_ids']
truncated_transcript = tokenizer.decode(input_ids, skip_special_tokens=True)

prompt = f"[INST] <<SYS>>\n{system_message}\n<</SYS>>\n\n{truncated_transcript} [/INST]"

# Run generation using max_new_tokens
gen = pipeline('text-generation', model=model, tokenizer=tokenizer)
result = gen(
    prompt,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.2
)

# Print the generated SOAP note
print(result[0]['generated_text'].replace(prompt, ''))


Subjective:
- Chief Complaint: Shortness of Breath; Deterioration over several weeks; Progressively Worsening Over Last Week; Feeling "Drowned"; Leg Swelling (Ankles); Chest Pain; Palpable Heart Rhythm Abnormality; Fainting Spells; Cough Linked to Dyspnea; Dry Cough Mostly; Dysphagia; Dyspepsia; Gross Obstruction of Small Bowel; History of Cardiomyopathy; Lack of Exercise; Diabetes; Hypertension; Prior Stroke; Poststroke Deafness; Sleep Apnea; Night Awakenings; Severe Anxiety; Nonspecific Symptoms; Unclear etiology; Presenting With Acute Dyspnea and Diffuse Swelling; Relevant Past History: Chronic stable hypertension; Type II diabetes; Left ventricular dysfunction; Prior stroke; Prior transcranial Doppler; Prior acute coronary syndrome; Prior mild trauma; Prior history of severe angina; Prior history of nausea and vomiting; Prior history of orthostatic hypotension; Prior history of migraine headaches; Prior history of intermittent claudicia; Prior history of exercise intolerance; Prior